In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# CELL 0 — Shared prep config  (run once; used by ALL 3 datasets)

In [2]:
import os, re, random, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- shared output contract (identical for every dataset) ----
TARGET_SIZE  = 256                       # materialize letterboxed to 256²; online random-crop to 224² at train time
PAD_COLOR    = (0, 0, 0)                  # letterbox fill
IMG_EXTS     = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
OUT_ROOT     = Path("/kaggle/working/prepared")   # cleaned dataset built here, then published as a Kaggle Dataset
SPLIT_RATIOS = {"train": 0.80, "val": 0.10, "test": 0.10}   # used when a source ships no split (rice)

# the manifest schema every dataset must emit, so the three merge cleanly:
MANIFEST_COLS = ["src_path", "filename", "label", "source_dataset", "split", "group_id"]

INPUT_ROOT = "/kaggle/input"
print("Mounted datasets:", os.listdir(INPUT_ROOT))
print("Output root     :", OUT_ROOT)

Mounted datasets: ['datasets']
Output root     : /kaggle/working/prepared


# CELL S1 — Rice Set 2: locate + manifest + normalize labels

In [3]:
RICE_LABEL_MAP = {
    "bacterialblight": "bacterial_blight", "brownspot": "brown_spot",
    "blast": "blast", "tungro": "tungro",
    "bacterial leaf blight": "bacterial_blight", "brown spot": "brown_spot",
    "leaf smut": "leaf_smut",
}
def rice_key(raw):
    k = raw.strip().lower()
    if k not in RICE_LABEL_MAP:
        raise KeyError(f"Unmapped rice label: {raw!r} — add it to RICE_LABEL_MAP")
    return RICE_LABEL_MAP[k]

cands = [os.path.join(INPUT_ROOT, d) for d in os.listdir(INPUT_ROOT)]
DATA_ROOT = next((c for c in cands if "rice" in c.lower()), cands[0])
print("Rice S2 DATA_ROOT =", DATA_ROOT)

rows = []
for p in Path(DATA_ROOT).rglob("*"):
    if p.is_file() and p.suffix.lower() in IMG_EXTS:
        rows.append({"src_path": str(p), "filename": p.name,
                     "raw_label": p.parent.name, "split": "all"})
s2 = pd.DataFrame(rows)
s2["key"]            = s2["raw_label"].map(rice_key)
s2["label"]         = "rice__" + s2["key"]
s2["source_dataset"] = "rice_s2_vbookshelf"

print("Total images:", len(s2), "(expect 120)")
print("Raw labels:", sorted(s2["raw_label"].unique()))
print("\nPer-class counts:")
print(s2["label"].value_counts().sort_index())

Rice S2 DATA_ROOT = /kaggle/input/datasets
Total images: 120 (expect 120)
Raw labels: ['Bacterial leaf blight', 'Brown spot', 'Leaf smut']

Per-class counts:
label
rice__bacterial_blight    40
rice__brown_spot          40
rice__leaf_smut           40
Name: count, dtype: int64


# CELL S2 — Rice Set 2: confirm zero-dup + measure the aspect-ratio problem

In [4]:
import hashlib
def _dct_matrix(N):
    n = np.arange(N); k = n.reshape(-1, 1)
    M = np.sqrt(2.0/N)*np.cos(np.pi*(2*n+1)*k/(2*N)); M[0,:] /= np.sqrt(2.0); return M
_D32 = _dct_matrix(32)
def phash64(im):
    g = im.convert("L").resize((32,32), Image.BILINEAR)
    d = _D32 @ np.asarray(g, dtype=np.float64) @ _D32.T
    flat = d[:8,:8].flatten(); med = np.median(flat[1:]); h = np.uint64(0)
    for b in (flat > med): h = (h << np.uint64(1)) | np.uint64(bool(b))
    return h

md5s, phs, dims, bad = [], [], [], 0
for p in s2["src_path"]:
    try:
        with open(p,"rb") as f: md5s.append(hashlib.md5(f.read()).hexdigest())
        with Image.open(p) as im:
            phs.append(int(phash64(im))); dims.append(im.size)
    except Exception:
        md5s.append(None); phs.append(None); dims.append((None,None)); bad += 1

s2["file_md5"], s2["phash"] = md5s, phs
s2["w"] = [d[0] for d in dims]; s2["h"] = [d[1] for d in dims]
s2["aspect"] = (s2["w"] / s2["h"]).round(2)
# every image is its own group (dataset is pristine); stamp for pipeline symmetry
s2["group_id"] = [f"s2_solo{i:04d}" for i in range(len(s2))]

print("Corrupt:", bad, "| exact dup files:",
      int(s2['file_md5'].duplicated(keep=False).sum()))
print("Color modes present:", end=" ")
print(pd.Series([Image.open(p).mode for p in s2['src_path']]).value_counts().to_dict())
print("\nAspect-ratio spread:")
print(s2["aspect"].describe().round(2).to_string())
print("\nHow many are extreme panoramas (aspect > 2.5)?",
      int((s2["aspect"] > 2.5).sum()), "of", len(s2))

Corrupt: 0 | exact dup files: 0
Color modes present: {'RGB': 120}

Aspect-ratio spread:
count    120.00
mean       3.31
std        0.66
min        1.25
25%        3.43
50%        3.43
75%        3.43
max        5.30

How many are extreme panoramas (aspect > 2.5)? 107 of 120


# CELL S3 — Rice Set 2: letterbox to 256² + write images + manifest

In [5]:
OUT_IMG = OUT_ROOT / "rice_s2" / "images"
OUT_IMG.mkdir(parents=True, exist_ok=True)

records, n_fail = [], 0
for i, r in enumerate(s2.itertuples(index=False)):
    try:
        with Image.open(r.src_path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im = ImageOps.pad(im, (TARGET_SIZE, TARGET_SIZE),
                              method=Image.BILINEAR, color=PAD_COLOR)   # aspect-preserving letterbox
            fn = f"{r.key}_{i:04d}.jpg"
            im.save(OUT_IMG / fn, "JPEG", quality=92)
        records.append({"src_path": r.src_path, "filename": fn, "label": r.label,
                        "source_dataset": "rice_s2_vbookshelf", "split": "unassigned",
                        "group_id": r.group_id})
    except Exception:
        n_fail += 1

man = pd.DataFrame(records, columns=MANIFEST_COLS)
man.to_csv(OUT_ROOT / "rice_s2" / "manifest.csv", index=False)
print("Written:", len(man), "| failed:", n_fail)
print("\nFinal per-class:")
print(man["label"].value_counts().sort_index())

Written: 120 | failed: 0

Final per-class:
label
rice__bacterial_blight    40
rice__brown_spot          40
rice__leaf_smut           40
Name: count, dtype: int64
